In [1]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "4"
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"
os.environ["VECLIB_MAXIMUM_THREADS"] = "4"

In [2]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer

/Users/yash/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/yash/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
articles = pd.read_csv("h-and-m-personalized-fashion-recommendations/articles.csv")

In [6]:
text_cols = [
    "prod_name",
    "product_type_name",
    "product_group_name",
    "graphical_appearance_name",
    "colour_group_name",
    "perceived_colour_master_name",
    "index_name",
    "garment_group_name",
    "detail_desc",
]

In [7]:
text_cols = [c for c in text_cols if c in articles.columns]
 
def build_description(row):
    """
    Concatenate the selected metadata fields into one text string per article.
    This string is what gets embedded — the richer and more distinct the fields,
    the better the resulting vector captures what the item actually is.
    """
    parts = [str(row[c]) for c in text_cols if pd.notna(row[c])]
    return " ".join(parts)

print("Building descriptions...")
articles["description"] = articles.apply(build_description, axis=1)

Building descriptions...


In [8]:
print("Loading embedding model...")
model = SentenceTransformer("all-MiniLM-L6-v2")  # 384-dim, fast, good enough for MVP
 
print(f"Embedding {len(articles)} articles... this may take a few minutes")
embeddings = model.encode(
    articles["description"].tolist(),
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True,
)
 
print("Embedding shape:", embeddings.shape)  # (num_articles, 384)

Loading embedding model...
Embedding 105542 articles... this may take a few minutes


Batches: 100%|██████████| 413/413 [01:44<00:00,  3.94it/s]


Embedding shape: (105542, 384)


In [9]:
out = pd.DataFrame({
    "article_id": articles["article_id"],
    "embedding": embeddings.tolist(),
})
out.to_parquet("article_embeddings.parquet", index=False)
print("Saved article_embeddings.parquet")
 
# ---- 5. Compute the initial global average vector ----
# This becomes global_state.avg_vector — the starting point for every fresh profile
global_avg = embeddings.mean(axis=0)
np.save("global_avg_vector.npy", global_avg)
print("Saved global_avg_vector.npy — this seeds every new profile")

Saved article_embeddings.parquet
Saved global_avg_vector.npy — this seeds every new profile


In [10]:
emb_df = pd.read_parquet("article_embeddings.parquet")
print(emb_df.shape)                    # should be (~105542, 2)
print(len(emb_df["embedding"].iloc[0])) # should be 384
print(articles["description"].sample(5).tolist())

(105542, 2)
384
['Billie. T-shirt Garment Upper body Melange Blue Blue Ladieswear Jersey Basic Short-sleeved, fitted jersey top with a slightly deeper neckline.', 'Phoebe Push Melbourne Bra Underwear Solid Black Black Lingeries/Tights Under-, Nightwear Push-up bra in microfibre with decorative straps at the front and underwired, moulded, padded cups for a larger bust and fuller cleavage. Adjustable shoulder straps and a hook-and-eye fastening at the back.', 'Evelina dress s/s Dress Garment Full body All over pattern Light Orange Orange Baby Sizes 50-98 Jersey Fancy Sleeveless dress in soft, patterned cotton jersey with a small opening and button at the back of the neck and flounces down the front that continue over the shoulders and down the back. Gathered seam at the top and a flared skirt.', 'Cleveland Regular Chino Trousers Garment Lower body Solid Beige Beige Menswear Trousers Chinos in stretch cotton twill with a zip fly and button, diagonal side pockets, a coin pocket and welt ba